# Schema Export — Genealogy Database

Exports the full column-level schema of the `genealogy` database to a Google Doc,
so Claude (via the Claude Project Google Drive connector) always has access to
current table/column names without requiring manual DESCRIBE queries.

## First run
1. Run all cells. A new Google Doc is created and the Doc ID is printed at the end.
2. Store the Doc ID in Databricks Secrets: `databricks secrets put-secret genealogy schema_export_doc_id --string-value <DOC_ID>`
3. Share the Doc link in your Claude Project context (read-only).

## Subsequent runs
Run all cells — the notebook detects the existing Doc ID from secrets and overwrites
the content in-place. The Doc URL never changes.

## When to re-run
After any session that adds, alters, or drops tables or views. Takes ~30 seconds.

In [0]:
# ── Cell 1: Install Google API client libraries ────────────────────────────────
# These are usually pre-installed on Databricks, but pip install ensures correct
# versions are available on the serverless cluster.
%pip install --quiet google-auth google-auth-httplib2 google-api-python-client

In [0]:
# ── Cell 2: Load OAuth credentials ────────────────────────────────────────────
# Reads Google OAuth credentials from Databricks Secrets (scope: genealogy).
# These are the same credentials used by the Render app (GOOGLE_OAUTH_* env vars).
#
# If you haven't stored them in secrets yet, run these once in your terminal:
#   databricks secrets put-secret genealogy google_oauth_client_id     --string-value <your_value>
#   databricks secrets put-secret genealogy google_oauth_client_secret --string-value <your_value>
#   databricks secrets put-secret genealogy google_oauth_refresh_token --string-value <your_value>
#
# Alternatively, set FALLBACK_* variables below for a one-off interactive run.

FALLBACK_CLIENT_ID     = ""   # leave blank to require secrets
FALLBACK_CLIENT_SECRET = ""
FALLBACK_REFRESH_TOKEN = ""

def _get_secret(scope, key, fallback=""):
    try:
        return dbutils.secrets.get(scope=scope, key=key)
    except Exception:
        return fallback

CLIENT_ID     = _get_secret("genealogy", "google_oauth_client_id",     FALLBACK_CLIENT_ID)
CLIENT_SECRET = _get_secret("genealogy", "google_oauth_client_secret", FALLBACK_CLIENT_SECRET)
REFRESH_TOKEN = _get_secret("genealogy", "google_oauth_refresh_token", FALLBACK_REFRESH_TOKEN)

# Also read the existing schema doc ID (populated after first run)
EXISTING_DOC_ID = _get_secret("genealogy", "schema_export_doc_id", "")

if not all([CLIENT_ID, CLIENT_SECRET, REFRESH_TOKEN]):
    raise ValueError(
        "Google OAuth credentials not found. "
        "Store them in Databricks Secrets (scope: genealogy) or set the FALLBACK_* variables above."
    )

print(f"OAuth credentials loaded.")
print(f"Existing doc ID: {EXISTING_DOC_ID or '(none — first run)'}")

In [0]:
# ── Cell 3: Query information_schema.columns ───────────────────────────────────
# Fetches all tables and columns in the genealogy catalog/database.
# information_schema is a standard SQL view available on all Databricks Unity Catalog
# setups — no warehouse spin-up needed for metadata queries.
#
# We exclude system/internal tables and include column comments where set.

schema_df = spark.sql("""
    SELECT
        table_schema,
        table_schema || '.' || table_name as table_name,
        column_name,
        full_data_type   AS data_type,
        comment
    FROM workspace.information_schema.columns
    WHERE table_schema in ('genealogy', 'staging_google_drive')
      AND table_name NOT LIKE '__apply_changes%'   -- exclude CDC internal tables
    ORDER BY table_schema, table_name, ordinal_position
""")

rows = schema_df.collect()
table_names = sorted(set(r.table_name for r in rows))

print(f"Found {len(table_names)} tables/views with {len(rows)} total columns.")
print("Tables:", ", ".join(table_names))

In [0]:
# ── Cell 4: Build markdown document ───────────────────────────────────────────
# Formats the schema as readable markdown that Claude can parse easily.
# Structure: top-level header → one ## section per table → column list.

from datetime import datetime, timezone

# Group columns by table
from collections import defaultdict
table_columns = defaultdict(list)
for r in rows:
    table_columns[r.table_name].append((r.column_name, r.data_type, r.comment or ""))

# Build markdown
now_utc = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
lines = [
    "# Genealogy Database Schema",
    "",
    f"**Catalog:** `workspace`  ",
    f"**Last exported:** {now_utc}  ",
    f"**Tables/views:** {len(table_names)}  ",
    f"**Total columns:** {len(rows)}",
    "",
    "---",
    "",
]

for table in sorted(table_columns.keys()):
    lines.append(f"## {table}")
    lines.append("")
    for col_name, data_type, comment in table_columns[table]:
        comment_str = f" — {comment}" if comment else ""
        lines.append(f"- `{col_name}` ({data_type}){comment_str}")
    lines.append("")

schema_markdown = "\n".join(lines)

# Preview first 600 chars
print(schema_markdown[:600])
print(f"\n... (total {len(schema_markdown):,} characters)")

In [0]:
# ── Cell 5: Write to Google Drive ─────────────────────────────────────────────
# Replicates the pattern from services/google_drive.py and routers/story.py:
#   - MediaInMemoryUpload plain text -> Google Doc conversion
#   - drive.files().create() for first run
#   - drive.files().update() to overwrite content on subsequent runs
#
# The doc is created in your root Drive (no parent folder set) — move it manually
# to wherever you like after the first run. The ID is stable across moves.

from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaInMemoryUpload

DOC_TITLE = "Genealogy DB Schema (auto-exported)"
GOOGLE_DRIVE_FOLDER = "Family Tree"

# Build credentials — same pattern as get_google_services() in google_drive.py
scopes = ["https://www.googleapis.com/auth/drive"]
creds = Credentials(
    token=None,
    refresh_token=REFRESH_TOKEN,
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    token_uri="https://oauth2.googleapis.com/token",
    scopes=scopes,
)
creds.refresh(Request())   # eager refresh — surfaces auth errors early

drive = build("drive", "v3", credentials=creds)

# Encode content as plain text — Drive converts to Google Doc automatically
media = MediaInMemoryUpload(
    schema_markdown.encode("utf-8"),
    mimetype="text/plain",
    resumable=False,
)

if EXISTING_DOC_ID:
    # ── Subsequent runs: overwrite existing doc content in-place ────────────
    # Using files().update() preserves the Doc ID and URL — Claude's reference
    # to the doc remains valid forever.
    drive.files().update(
        fileId=EXISTING_DOC_ID,
        body={"name": DOC_TITLE},
        media_body=media,
    ).execute()
    doc_id  = EXISTING_DOC_ID
    doc_url = f"https://docs.google.com/document/d/{doc_id}/edit"
    print(f"✅ Schema doc updated in-place.")

else:
    # ── First run: create new Google Doc ────────────────────────────────────
    file_meta = drive.files().create(
        body={
            "name":     DOC_TITLE,
            "mimeType": "application/vnd.google-apps.document",
            "parents":  GOOGLE_DRIVE_FOLDER,
        },
        media_body=media,
        fields="id",
    ).execute()
    doc_id  = file_meta["id"]
    doc_url = f"https://docs.google.com/document/d/{doc_id}/edit"
    print(f"✅ Schema doc created (first run).")
    print()
    print(f"▶ IMPORTANT — store this Doc ID in Databricks Secrets so future runs overwrite in-place:")
    print(f"  databricks secrets put-secret genealogy schema_export_doc_id --string-value {doc_id}")
    print()
    print(f"▶ Add this Doc ID to your Claude Project context so Claude can read it automatically:")
    print(f"  Doc ID: {doc_id}")

print()
print(f"📄 Doc URL: {doc_url}")
print(f"   Tables exported: {len(table_names)}")
print(f"   Columns exported: {len(rows)}")

## After first run — setup checklist

1. **Store the Doc ID** in Databricks Secrets (command printed above) so subsequent runs overwrite in-place.
2. **Open the Doc URL** and verify it looks correct — one section per table, columns with types.
3. **Add the Doc to your Claude Project** — paste the Doc URL or ID into the Project's file/context panel. Claude will read it automatically at the start of each session.
4. **Re-run this notebook** after any session that changes schema (new tables, new columns, ALTER TABLE).

## Scheduling (optional)

You can attach this notebook to a Databricks Job with a weekly schedule if you want it to refresh automatically. However, manual re-runs after schema-changing sessions are probably sufficient given how infrequently the schema changes.